# Train Procrastination Model from feature_matrix.csv

This notebook loads the pre-built feature matrix from ml/feature_matrix.csv (generated by ml/feature_engineering.py), trains classification models, and evaluates performance.

**Features:** 11 columns | **Target:** procrastination_label (0/1)

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)
import joblib
import xgboost as xgb

In [2]:
# Path to feature_matrix.csv
for base in [".", "..", "../.."]:
    path = os.path.join(os.path.abspath(base), "ml", "feature_matrix.csv")
    if os.path.isfile(path):
        break
else:
    raise FileNotFoundError(
        "ml/feature_matrix.csv not found. Run python ml/feature_engineering.py first."
    )

df = pd.read_csv(path)
print("Loaded:", path)
print("Shape:", df.shape)
print("Columns:", list(df.columns))

Loaded: d:\workspace\2025_2026\Second term\KKU\CS\LMS\LMS_SYSTEM\ml\feature_matrix.csv
Shape: (21190, 13)
Columns: ['id_student', 'courses_count', 'assignments_count', 'workload_level', 'activity_count_in_enrolled_courses', 'previous_delays_count', 'non_submission_count', 'course_engagement', 'last7d_engagement', 'last7d_unique_resources', 'days_to_first_activity', 'num_of_prev_attempts', 'procrastination_label']


In [3]:
# EDA: explore the dataset to know more about it
print("Shape:", df.shape)
print("\nLabel distribution:")
print(df["procrastination_label"].value_counts())

# Use cleaned copy for describe (avoid inf/nan breaking stats and warnings)
df_clean = df.replace([np.inf, -np.inf], np.nan)
feature_cols = [c for c in df.columns if c not in ("id_student", "procrastination_label")]
df_clean = df_clean.copy()
df_clean[feature_cols] = df_clean[feature_cols].fillna(0)
print("\nBasic stats (inf/nan replaced for display):")
print(df_clean.describe())

print("\nMissing values (raw):")
print(df.isna().sum())
print("\nid_student max:", df["id_student"].max())
print("id_student count:", df["id_student"].count())


Shape: (21190, 13)

Label distribution:
procrastination_label
0    11220
1     9970
Name: count, dtype: int64

Basic stats (inf/nan replaced for display):
         id_student  courses_count  assignments_count  workload_level  \
count  2.119000e+04   21190.000000       21190.000000    21190.000000   
mean   7.077830e+05       1.062813          10.547853        0.975572   
std    5.545166e+05       0.244376           3.569162        0.378299   
min    6.516000e+03       1.000000           5.000000        0.339167   
25%    5.055655e+05       1.000000           7.000000        0.649167   
50%    5.891880e+05       1.000000          12.000000        0.929773   
75%    6.432552e+05       1.000000          13.000000        1.231190   
max    2.710343e+06       3.000000          36.000000        1.500000   

       activity_count_in_enrolled_courses  previous_delays_count  \
count                        21190.000000           21190.000000   
mean                           324.015715          

In [38]:
df.corr()

,id_student,courses_count,assignments_count,workload_level,activity_count_in_enrolled_courses,previous_delays_count,non_submission_count,course_engagement,last7d_engagement,last7d_unique_resources,days_to_first_activity,num_of_prev_attempts,procrastination_label
id_student,1.000000,-0.018270,-0.039576,-0.015723,-0.008881,0.001831,-0.026023,0.030828,0.027138,0.030903,-0.005377,0.004286,0.003727
courses_count,-0.018270,1.000000,0.538856,-0.017922,0.018262,0.144137,0.075199,0.174845,0.224682,0.223604,-0.031177,0.022036,0.221992
assignments_count,-0.039576,0.538856,1.000000,0.560151,0.483461,0.287496,0.230646,0.199804,0.169116,0.195319,-0.011473,0.076325,0.304301
workload_level,-0.015723,-0.017922,0.560151,1.000000,0.960866,0.017077,0.130651,0.244126,0.164863,0.263626,-0.050703,0.114743,-0.097237
activity_count_in_enrolled_courses,-0.008881,0.018262,0.483461,0.960866,1.000000,0.058189,0.105754,0.233955,0.189204,0.288512,-0.064526,0.126998,-0.044547
previous_delays_count,0.001831,0.144137,0.287496,0.017077,0.058189,1.000000,-0.191428,0.152112,0.193327,0.180182,-0.133860,0.064803,0.513601
non_submission_count,-0.026023,0.075199,0.230646,0.130651,0.105754,-0.191428,1.000000,-0.572614,-0.655862,-0.626547,0.288598,0.136711,-0.046534
course_engagement,0.030828,0.174845,0.199804,0.244126,0.233955,0.152112,-0.572614,1.000000,0.839794,0.800187,-0.514261,-0.116183,0.056566
last7d_engagement,0.027138,0.224682,0.169116,0.164863,0.189204,0.193327,-0.655862,0.839794,1.000000,0.949889,-0.329366,-0.106668,0.134836
last7d_unique_resources,0.030903,0.223604,0.195319,0.263626,0.288512,0.180182,-0.626547,0.800187,0.949889,1.000000,-0.324490,-0.088724,0.097299


In [41]:
FEATURE_COLS = [
    "courses_count",
    "assignments_count",
    "workload_level",
    "activity_count_in_enrolled_courses",
    "previous_delays_count",
    "non_submission_count",
    "course_engagement",
    "last7d_engagement",
    #"last7d_unique_resources",
    "days_to_first_activity",
    #"num_of_prev_attempts",
]

X = df[FEATURE_COLS].fillna(0)
y = df["procrastination_label"]

print("Label distribution:")
print(y.value_counts())
print(df.head())

Label distribution:
procrastination_label
0    11220
1     9970
Name: count, dtype: int64
   id_student  courses_count  assignments_count  workload_level  \
0       11391              1                  6        0.594167   
1       28400              1                  6        0.594167   
2       31604              1                  6        0.594167   
3       32885              1                  6        0.594167   
4       38053              1                  6        0.594167   

   activity_count_in_enrolled_courses  previous_delays_count  \
0                               211.0                    1.0   
1                               211.0                    3.0   
2                               211.0                    3.0   
3                               211.0                    4.0   
4                               211.0                    5.0   

   non_submission_count  course_engagement  last7d_engagement  \
0                     0           4.619478           2.21

In [42]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print("Train:", X_train.shape[0], "Test:", X_test.shape[0])

Train: 16952 Test: 4238


In [ ]:
# (2) Class weights for imbalanced recall
scale_pos = (y_train == 0).sum() / max((y_train == 1).sum(), 1)

base_models = [
    (
        "Logistic Regression",
        LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    ),
    (
        "Random Forest",
        RandomForestClassifier(
            n_estimators=200, class_weight="balanced", random_state=42
        ),
    ),
    (
        "XGBoost",
        xgb.XGBClassifier(
            eval_metric="logloss", scale_pos_weight=scale_pos, random_state=42
        ),
    ),
]

# (3) 5-fold cross-validation
print("5-fold CV F1 scores:")
for name, model in base_models:
    scores = cross_val_score(model, X_train_s, y_train, cv=5, scoring="f1", n_jobs=-1)
    print(f"  {name}: {scores.mean():.4f} (+/- {scores.std():.4f})")

# (1) Hyperparameter tuning for RF and XGBoost
print("\nGridSearchCV (RF)...")
rf_grid = GridSearchCV(
    RandomForestClassifier(class_weight="balanced", random_state=42),
    param_grid={
        "n_estimators": [150, 250, 350],
        "max_depth": [12, 18, None],
        "min_samples_leaf": [2, 4, 6],
        "min_samples_split": [2, 5, 10],
        "max_features": ["sqrt", "log2"],
    },
    cv=5,
    scoring="f1",
    n_jobs=-1,
    verbose=0,
)
rf_grid.fit(X_train_s, y_train)
best_rf = rf_grid.best_estimator_
print(f"  Best RF params: {rf_grid.best_params_}")
#  Best RF params: {'max_depth': 18, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 10, 'n_estimators': 350}


print("GridSearchCV (XGBoost)...")
xgb_grid = GridSearchCV(
    xgb.XGBClassifier(
        eval_metric="logloss", scale_pos_weight=scale_pos, random_state=42
    ),
    param_grid={
        "n_estimators": [200, 300],
        "max_depth": [5, 7],
        "learning_rate": [0.05, 0.1],
        "gamma": [0, 0.1, 0.5],
        "reg_alpha": [0, 0.1],
        "min_child_weight": [1, 3],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0],
    },
    cv=5,
    scoring="f1",
    n_jobs=-1,
    verbose=0,
)
xgb_grid.fit(X_train_s, y_train)
best_xgb = xgb_grid.best_estimator_
print(f"  Best XGB params: {xgb_grid.best_params_}")
#  Best XGB params: {'colsample_bytree': 0.8, 'gamma': 0, 'learning_rate': 0.05, 'max_depth': 5, 'min_child_weight': 1, 'n_estimators': 300, 'reg_alpha': 0.1, 'subsample': 0.8}

# Fit LR (no grid), use tuned RF and XGB
lr = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
lr.fit(X_train_s, y_train)

models = [
    ("Logistic Regression", lr),
    ("Random Forest", best_rf),
    ("XGBoost", best_xgb),
]

results = {}
for name, model in models:
    pred = model.predict(X_test_s)
    results[name] = {
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0),
        "confusion_matrix": confusion_matrix(y_test, pred).tolist(),
    }
    print(
        f"{name}: F1={results[name]['f1']:.4f} Acc={results[name]['accuracy']:.4f} Recall={results[name]['recall']:.4f}"
    )

In [43]:
# Best params from GridSearchCV (use to skip GridSearch for faster runs)
#  Best RF params: {'max_depth': 18, 'max_features': 'sqrt', 'min_samples_leaf': 2, 
# 'min_samples_split': 10, 'n_estimators': 350}

BEST_RF_PARAMS = {
    "max_depth": 18,
    "max_features": "sqrt",
    "min_samples_leaf": 2,
    "min_samples_split": 10,
    "n_estimators": 350,
}
BEST_XGB_PARAMS = {
    "colsample_bytree": 0.8,
    "learning_rate": 0.05,
    "max_depth": 5,
    "min_child_weight": 1,
    "n_estimators": 300,
    "reg_alpha": 0.1,
    "gamma": 0,
    "subsample": 0.8,
}
#  Best XGB params: {'colsample_bytree': 0.8, 'gamma': 0, 'learning_rate': 0.05, 'max_depth': 5,
#  'min_child_weight': 1, 'n_estimators': 300, 'reg_alpha': 0.1, 'subsample': 0.8}

# Use best params directly (no GridSearch)
scale_pos = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
best_rf = RandomForestClassifier(
    class_weight="balanced", random_state=42, **BEST_RF_PARAMS
)
best_xgb = xgb.XGBClassifier(
    eval_metric="logloss",
    scale_pos_weight=scale_pos,
    random_state=42,
    **BEST_XGB_PARAMS,
)
best_rf.fit(X_train_s, y_train)
best_xgb.fit(X_train_s, y_train)

lr = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
lr.fit(X_train_s, y_train)

models = [
    ("Logistic Regression", lr),
    ("Random Forest", best_rf),
    ("XGBoost", best_xgb),
]

results = {}
for name, model in models:
    pred = model.predict(X_test_s)
    results[name] = {
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0),
        "confusion_matrix": confusion_matrix(y_test, pred).tolist(),
    }
    print(
        f"{name}: F1={results[name]['f1']:.4f} Acc={results[name]['accuracy']:.4f} Recall={results[name]['recall']:.4f}"
    )

Logistic Regression: F1=0.7856 Acc=0.8042 Recall=0.7628
Random Forest: F1=0.8361 Acc=0.8525 Recall=0.7994
XGBoost: F1=0.8385 Acc=0.8544 Recall=0.8034


In [30]:
best_name = max(results, key=lambda k: results[k]["f1"])
print(f"Best: {best_name} (F1={results[best_name]['f1']:.4f})")

Best: XGBoost (F1=0.8391)


In [31]:
out_dir = os.path.join(os.path.dirname(os.path.abspath(path)), "model")
os.makedirs(out_dir, exist_ok=True)

# Save evaluation report
eval_path = os.path.join(out_dir, "evaluation_report.txt")
with open(eval_path, "w") as f:
    f.write("Procrastination model â€“ evaluation (feature_matrix.csv, 80/20 split)\n")
    f.write("=" * 60 + "\n\n")
    for name, r in results.items():
        f.write(f"{name}\n")
        f.write(f"  Accuracy:  {r['accuracy']:.4f}\n")
        f.write(f"  Precision: {r['precision']:.4f}\n")
        f.write(f"  Recall:    {r['recall']:.4f}\n")
        f.write(f"  F1:        {r['f1']:.4f}\n")
        f.write(f"  Confusion matrix (TN FP / FN TP): {r['confusion_matrix']}\n\n")
    f.write("=" * 60 + "\n")
    f.write(f"Best model (by F1): {best_name}\n")
    f.write(f"Features: {len(FEATURE_COLS)}\n")
print("Saved evaluation to", eval_path)

# Save model
best_model = next(m for n, m in models if n == best_name)
best_model.fit(scaler.fit_transform(X), y)
joblib.dump(
    {"model": best_model, "scaler": scaler, "feature_order": FEATURE_COLS},
    os.path.join(out_dir, "procrastination_model.joblib"),
)
print("Saved model to", out_dir)

Saved evaluation to d:\workspace\2025_2026\Second term\KKU\CS\LMS\LMS_SYSTEM\ml\model\evaluation_report.txt
Saved model to d:\workspace\2025_2026\Second term\KKU\CS\LMS\LMS_SYSTEM\ml\model


In [32]:
# Print feature importances if supported by the best model
import numpy as np

if hasattr(best_model, "feature_importances_"):
    importances = best_model.feature_importances_
    indices = np.argsort(importances)[::-1]
    print("\nFeature Importances:")
    for idx in indices:
        print(f"{FEATURE_COLS[idx]}: {importances[idx]:.4f}")
elif hasattr(best_model, "coef_"):
    coefs = np.abs(best_model.coef_[0])
    indices = np.argsort(coefs)[::-1]
    print("\nFeature Importances (absolute coefficients):")
    for idx in indices:
        print(f"{FEATURE_COLS[idx]}: {coefs[idx]:.4f}")
else:
    print("Feature importances not available for this model.")


Feature Importances:
previous_delays_count: 0.2982
assignments_count: 0.1994
activity_count_in_enrolled_courses: 0.1978
courses_count: 0.1050
workload_level: 0.0805
non_submission_count: 0.0465
last7d_engagement: 0.0397
course_engagement: 0.0330
